# 06_prediction — 予測・Kelly 基準買い目生成

本番エンドポイント (`python-api/routers/predict.py`) と同一ロジック:
- モデルロード → 確率推定
- Kelly 基準で馬券推奨
- INV-02: `odds is not None` チェック

出力:
- `reports/prediction.csv`

In [1]:
import sys, json
from pathlib import Path
_NB_DIR = Path().resolve()
if _NB_DIR.name != 'notebooks': _NB_DIR = _NB_DIR.parent
if str(_NB_DIR) not in sys.path: sys.path.insert(0, str(_NB_DIR))
from utils.nb_config import *

_cfg_file = NB_DIR / 'config.json'
if _cfg_file.exists():
    _cfg = json.loads(_cfg_file.read_text(encoding='utf-8'))
    for _k in ('TARGET','RANDOM_STATE','TEST_START','TEST_END'):
        if _k in _cfg: globals()[_k] = _cfg[_k]

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import joblib

# モデルロード
_bundle = joblib.load(MODELS_DIR / 'lgb_model.pkl')
_clf         = _bundle['model']
_optimizer   = _bundle['optimizer']
_cat_features= _bundle['cat_features']
_feature_cols= _bundle['feature_cols']

# 特徴量データロード
_feat_cache = NB_DIR / 'data' / 'features' / 'features.pkl'
_data   = joblib.load(_feat_cache)
X_test  = _data['X_test']
y_test  = _data['y_test']
df_test = _data['df_test']

print(f"X_test: {X_test.shape}")

X_test: (36344, 146)


In [2]:
# ── Step1: 確率推定 ───────────────────────────────────────────
# テストデータを feature_cols に合わせる
X_pred = X_test.reindex(columns=_feature_cols, fill_value=0)

# object / category 列を数値コードに変換（LightGBM は int/float/bool のみ受け付ける）
def _encode_obj(df: pd.DataFrame) -> pd.DataFrame:
    out = {}
    for col in df.columns:
        s = df[col]
        if hasattr(s, 'cat'):
            codes = s.cat.codes.astype(np.float32)
            codes[codes == -1] = np.nan
            out[col] = codes
        elif s.dtype == object:
            codes_int, _ = pd.factorize(s)
            arr = codes_int.astype(np.float32)
            arr[arr == -1] = np.nan
            out[col] = pd.Series(arr, index=df.index)
        else:
            out[col] = s
    return pd.DataFrame(out, index=df.index)

X_pred = _encode_obj(X_pred)

# モデル種別を bundle の task キーで判定
_task = _bundle.get('task', 'binary')

if _task == 'regression':
    # 回帰モデル（speed_deviation など）
    # predict() で連続スコアを取得し、レース内 min-max 正規化で 0〜1 に変換
    _raw_score = _clf.predict(X_pred)

    if 'race_id' in df_test.columns:
        _race_id_arr = df_test['race_id'].reindex(X_test.index).values
        _score_series = pd.Series(_raw_score, index=X_test.index)
        _norm_scores = _score_series.copy().astype(float)
        for _rid in np.unique(_race_id_arr):
            _mask = _race_id_arr == _rid
            _idx = np.where(_mask)[0]
            _s = _score_series.iloc[_idx]
            _mn, _mx = _s.min(), _s.max()
            if _mx > _mn:
                _norm_scores.iloc[_idx] = (_s - _mn) / (_mx - _mn)
            else:
                _norm_scores.iloc[_idx] = 0.5
        win_proba = _norm_scores.values
    else:
        # レース情報がない場合は全体 min-max
        _mn, _mx = _raw_score.min(), _raw_score.max()
        win_proba = (_raw_score - _mn) / (_mx - _mn) if _mx > _mn else np.full_like(_raw_score, 0.5)
else:
    # 分類モデル（win / place3 など）
    win_proba = _clf.predict_proba(X_pred)[:, 1]

print(f"確率推定完了: {len(win_proba)} 件  (task={_task!r})")
print(f"平均スコア: {win_proba.mean():.4f}  最大: {win_proba.max():.4f}")


確率推定完了: 36344 件  (task='regression')
平均スコア: 0.4921  最大: 1.0000


In [3]:
# ── Step2: オッズ取得（INV-02 準拠）─────────────────────────
# 【設計】race_results_ultimate は 1行 = 1頭 の JSON レコード。
#         horse_number + race_id で突合してオッズを取得する。
# INV-02: is not None で判定（0.0 を falsy 扱いしない）

import sqlite3, json as _json

_db_path = get_db_path(DATA_SOURCE_MODE)
_odds_from_db = pd.Series(np.nan, index=df_test.index, name='odds')

try:
    with sqlite3.connect(_db_path) as _conn:
        _cur = _conn.cursor()

        # race_results_ultimate は 1行=1頭 の JSON
        # JSON 内に odds フィールドがあるかを SQLite の json_extract で直接取得
        _cur.execute(
            "SELECT race_id, json_extract(data,'$.horse_number'), json_extract(data,'$.odds') "
            "FROM race_results_ultimate "
            "WHERE json_extract(data,'$.odds') IS NOT NULL "
            "LIMIT 5"
        )
        _sample = _cur.fetchall()
        if _sample:
            print(f"json_extract サンプル: {_sample}")
            # 本番取得
            _race_ids_str = df_test['race_id'].astype(str).unique().tolist()
            _ph = ','.join(['?'] * len(_race_ids_str))
            _cur.execute(
                f"SELECT race_id, "
                f"CAST(json_extract(data,'$.horse_number') AS INTEGER), "
                f"CAST(json_extract(data,'$.odds') AS REAL) "
                f"FROM race_results_ultimate "
                f"WHERE race_id IN ({_ph}) "
                f"AND json_extract(data,'$.odds') IS NOT NULL",
                _race_ids_str
            )
            _rows = _cur.fetchall()
            print(f"  取得レコード数: {len(_rows):,}")
            if _rows:
                _odds_df = pd.DataFrame(_rows, columns=['race_id','horse_number','odds'])
                _odds_df = _odds_df.drop_duplicates(['race_id','horse_number'])
                print(f"  オッズ範囲: {_odds_df['odds'].min():.1f}〜{_odds_df['odds'].max():.1f}")

                _key = df_test[['race_id','horse_number']].copy()
                _key['race_id'] = _key['race_id'].astype(str)
                _key['horse_number'] = pd.to_numeric(_key['horse_number'], errors='coerce').astype('Int64')
                _odds_df['race_id'] = _odds_df['race_id'].astype(str)
                _odds_df['horse_number'] = _odds_df['horse_number'].astype('Int64')
                _merged = _key.merge(_odds_df, on=['race_id','horse_number'], how='left')
                _odds_from_db = pd.Series(_merged['odds'].values, index=df_test.index, name='odds')
        else:
            # json_extract が使えない → Python でパース
            print("json_extract で odds なし → Python パースを試行...")
            _cur.execute(
                f"SELECT race_id, data FROM race_results_ultimate "
                f"WHERE race_id IN ({','.join(['?']*len(_race_ids_str))})",
                _race_ids_str
            )
            _records = []
            for _rid, _blob in _cur.fetchall():
                try:
                    _d = _json.loads(_blob) if isinstance(_blob, str) else _blob
                    if isinstance(_d, dict):
                        _o = _d.get('odds')
                        _hn = _d.get('horse_number') or _d.get('horse_no')
                        if _o is not None and _hn is not None:
                            _records.append({'race_id': str(_rid),
                                             'horse_number': int(float(_hn)),
                                             'odds': float(_o)})
                except Exception:
                    pass
            if _records:
                _odds_df = pd.DataFrame(_records).drop_duplicates(['race_id','horse_number'])
                _key = df_test[['race_id','horse_number']].copy()
                _key['race_id'] = _key['race_id'].astype(str)
                _key['horse_number'] = pd.to_numeric(_key['horse_number'], errors='coerce').astype('Int64')
                _odds_df['horse_number'] = _odds_df['horse_number'].astype('Int64')
                _merged = _key.merge(_odds_df, on=['race_id','horse_number'], how='left')
                _odds_from_db = pd.Series(_merged['odds'].values, index=df_test.index, name='odds')
                print(f"  Python パース: {len(_records):,} レコード")
            else:
                print("  ⚠ オッズ列がデータ内に存在しません")

except Exception as _e:
    print(f"DB オッズ取得失敗: {type(_e).__name__}: {_e}")

# X_test と index を合わせる
_odds_aligned = _odds_from_db.reindex(X_test.index)

_valid_n = int(_odds_aligned.notna().sum())
if _valid_n > 0:
    print(f"\nオッズ取得完了: 有効={_valid_n:,}件  min={_odds_aligned.min():.1f}  max={_odds_aligned.max():.1f}")
else:
    print("\n⚠ オッズ 全NaN")
    print("  考えられる原因:")
    print("  1. race_results_ultimate の data JSON に 'odds' キーが存在しない")
    print("  2. テストデータ期間のレースはオッズ未スクレイプ")
    print("  → Kelly推奨は生成されません（実運用時はスクレイプ済みオッズが使われます）")


json_extract サンプル: [('202445010106', 4, 1.7), ('202445010106', 9, 6.4), ('202445010105', 6, 6.1), ('202445010108', 2, 479.9), ('202445010107', 4, 2.0)]


  取得レコード数: 102,179
  オッズ範囲: 1.0〜999.9

オッズ取得完了: 有効=36,075件  min=1.0  max=939.0


In [4]:
# ── Step3: Kelly 基準計算 ─────────────────────────────────────
# Kelly: f* = (p * (b - 1) - (1 - p)) / (b - 1)  where b = odds (日本式: 元返し込み)
# 実装: routers/predict.py の kelly_fraction 計算と同一

KELLY_FRACTION = 0.25   # Full Kelly の 1/4 推奨
MIN_ODDS       = 1.0
MAX_KELLY      = 0.20   # 最大賭け比率 20%

def kelly(p: float, odds: float, fraction: float = KELLY_FRACTION) -> float:
    """Kelly criterion (odds は倍率・元返し込み)"""
    if odds is None or np.isnan(odds) or odds <= MIN_ODDS:
        return 0.0
    b = odds - 1.0  # net odds
    f = (p * b - (1 - p)) / b
    return max(0.0, min(float(f * fraction), MAX_KELLY))

_kelly_fractions = np.array([
    kelly(p, o) for p, o in zip(win_proba, _odds_aligned.values)
])

_recommended = _kelly_fractions > 0
print(f"Kelly > 0 件数: {_recommended.sum()} / {len(_kelly_fractions)}")

Kelly > 0 件数: 26328 / 36344


In [5]:
# ── Step4: 予測結果 DataFrame 組立 ───────────────────────────
_id_cols = {}
for c in ('race_id', 'horse_id', 'horse_name'):
    if c in df_test.columns:
        _id_cols[c] = df_test[c].reindex(X_test.index).values

prediction_df = pd.DataFrame({
    **_id_cols,
    'win_probability': win_proba,
    'odds':            _odds_aligned.values,
    'kelly_fraction':  _kelly_fractions,
    'recommended':     _recommended,
    'actual':          y_test.values if y_test is not None else np.nan,
})

# レースごとに確率降順でソート
if 'race_id' in prediction_df.columns:
    prediction_df = (
        prediction_df
        .sort_values(['race_id', 'win_probability'], ascending=[True, False])
        .reset_index(drop=True)
    )

print(f"予測件数: {len(prediction_df):,}")
display(prediction_df.head(20))

予測件数: 36,344


,race_id,horse_id,horse_name,win_probability,odds,kelly_fraction,recommended,actual
0,201701020101,2015100719,ハピネスメーカー,1.000000,11.4,0.200000,True,0.707107
1,201701020101,2015106249,パワースピネル,0.000000,7.2,0.000000,False,-0.707107
2,201701020102,2015103195,キングキングキング,1.000000,3.4,0.200000,True,1.590738
3,201701020102,2015102478,ディアジラソル,0.483084,27.6,0.115913,True,0.139155
4,201701020102,2015103879,パスポート,0.467052,28.5,0.111918,True,-0.577514
5,201701020102,2015100153,クリノモリゾ,0.278248,113.0,0.067951,True,-1.051942
6,201701020102,2015102431,ファストライフ,0.000000,2.6,0.000000,False,-0.100418
7,201701020103,2014103304,メモリーフェイス,1.000000,2.2,0.200000,True,0.993326
8,201701020103,2014100829,エンパイアスタイル,0.886470,18.6,0.200000,True,-0.835745
9,201701020103,2014101455,ペイシャオブワキア,0.843606,4.7,0.200000,True,0.993326


In [6]:
# ── Step5: 保存 ───────────────────────────────────────────────
_out = REPORTS_DIR / 'prediction.csv'
prediction_df.to_csv(_out, index=False, encoding='utf-8-sig')
print(f"prediction.csv 保存: {_out}")

# 推奨馬券サマリ
_rec = prediction_df[prediction_df['recommended']]
print(f"\n推奨馬券: {len(_rec)} 件")
if len(_rec) > 0:
    display(_rec.head(10))

prediction.csv 保存: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\reports\prediction.csv

推奨馬券: 26328 件


,race_id,horse_id,horse_name,win_probability,odds,kelly_fraction,recommended,actual
0,201701020101,2015100719,ハピネスメーカー,1.000000,11.4,0.200000,True,0.707107
2,201701020102,2015103195,キングキングキング,1.000000,3.4,0.200000,True,1.590738
3,201701020102,2015102478,ディアジラソル,0.483084,27.6,0.115913,True,0.139155
4,201701020102,2015103879,パスポート,0.467052,28.5,0.111918,True,-0.577514
5,201701020102,2015100153,クリノモリゾ,0.278248,113.0,0.067951,True,-1.051942
7,201701020103,2014103304,メモリーフェイス,1.000000,2.2,0.200000,True,0.993326
8,201701020103,2014100829,エンパイアスタイル,0.886470,18.6,0.200000,True,-0.835745
9,201701020103,2014101455,ペイシャオブワキア,0.843606,4.7,0.200000,True,0.993326
10,201701020103,2014104107,サウンドベティ,0.808902,11.2,0.197542,True,0.001550
12,201701020104,2014100134,ペイシャデック,1.000000,6.8,0.200000,True,0.979541
